# S17 · Calculadora en Python · Notebook 02: reporte PDF

| | |
|---|---|
| **Programa** | IA Explorer – TCS / TecNM (CPFIA) |
| **Módulo** | 02 |
| **Sesión** | 17 · Python para Análisis de Datos |
| **Fecha** | 2026-11-12 |
| **Autor** | Luis Saúl González Guerra |

## 1. Objetivo

Generar el reporte en PDF de la práctica, en este orden:

1. Portada con los temas de la Sesión 17 y en qué parte de la calculadora usé cada uno.
2. Las capturas de pantalla que tomé mientras trabajaba en el notebook.
3. El notebook `01_calculadora.ipynb` completo, con su código y resultados.

Las capturas están en `outputs/capturas/` con un nombre fijo por paso. Si alguna no existe, su página se marca en rojo.

## 2. Configuración

In [1]:
import shutil
import subprocess
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import nbformat
from matplotlib.backends.backend_pdf import PdfPages
from nbconvert import HTMLExporter
from pypdf import PdfReader, PdfWriter

BASE_DIR = Path.cwd().parent
OUTPUT_DIR = BASE_DIR / "outputs"
CAPTURAS_DIR = OUTPUT_DIR / "capturas"
CAPTURAS_DIR.mkdir(parents=True, exist_ok=True)
PDF_OUT = OUTPUT_DIR / "S17_Calculadora_Python.pdf"
NOTEBOOK_01 = BASE_DIR / "notebooks" / "01_calculadora.ipynb"
CHROME_RUTAS = [
    "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome",
    "C:/Program Files/Google/Chrome/Application/chrome.exe",
    shutil.which("google-chrome") or "",
    shutil.which("chromium") or "",
]

AUTOR = "Luis Saúl González Guerra"
FECHA = "Noviembre 2026"
A4 = (11.69, 8.27)
AZUL, NARANJA, GRIS, TINTA = "#2F5D8A", "#E8704A", "#9AA5B1", "#1F2A36"

## 3. Capturas del proceso

Cada página del reporte corresponde a una captura, en el orden en que las tomé.

In [2]:
CAPTURAS = [
    ("Entorno", "Notebook abierto en VS Code con el entorno .venv del repositorio", "01_entorno"),
    ("Pruebas", "Pruebas de las siete operaciones", "02_pruebas"),
    ("Calculadora interactiva", "Sesión de uso de la calculadora", "03_calculadora_sesion"),
    ("Resumen", "Resumen de la sesión a partir del historial", "04_resumen_sesion"),
]


def buscar_captura(nombre):
    for ext in (".png", ".jpg", ".jpeg"):
        ruta = CAPTURAS_DIR / f"{nombre}{ext}"
        if ruta.exists():
            return ruta
    return None


faltan = 0
for _, titulo, clave in CAPTURAS:
    encontrada = buscar_captura(clave) is not None
    if not encontrada:
        faltan += 1
    print(f"{'Sí   ' if encontrada else 'Falta'}  {clave:<24} {titulo}")

Sí     01_entorno               Notebook abierto en VS Code con el entorno .venv del repositorio
Sí     02_pruebas               Pruebas de las siete operaciones
Sí     03_calculadora_sesion    Sesión de uso de la calculadora
Sí     04_resumen_sesion        Resumen de la sesión a partir del historial


## 4. Notebook 01 a PDF

El notebook 01 se convierte a HTML con `nbconvert` y Google Chrome lo imprime a PDF sin abrir ventana. Se convierte tal como quedó guardado, así que antes hay que ejecutarlo completo y guardarlo.

In [3]:
chrome = next((r for r in CHROME_RUTAS if r and Path(r).exists()), None)
if chrome is None:
    raise FileNotFoundError("No se encontró Google Chrome. Se puede agregar su ruta a CHROME_RUTAS.")

nb01 = nbformat.read(NOTEBOOK_01, as_version=4)
html, _ = HTMLExporter(template_name="lab").from_notebook_node(nb01)
estilo = ("<style>@page{size:A4;margin:12mm} body{font-size:11px}"
          " .jp-OutputArea-output pre, .jp-InputArea-editor pre{white-space:pre-wrap;word-break:break-word}"
          " .jp-RenderedHTMLCommon table{font-size:9px}</style></head>")
NOTEBOOK_HTML = OUTPUT_DIR / "_notebook01.html"
NOTEBOOK_PDF = OUTPUT_DIR / "_notebook01.pdf"
NOTEBOOK_HTML.write_text(html.replace("</head>", estilo, 1), encoding="utf-8")

subprocess.run([chrome, "--headless=new", "--disable-gpu", "--no-pdf-header-footer",
                f"--print-to-pdf={NOTEBOOK_PDF}", NOTEBOOK_HTML.as_uri()],
               check=True, capture_output=True, timeout=180)
paginas_notebook = len(PdfReader(NOTEBOOK_PDF).pages)
print(f"Notebook 01 convertido: {paginas_notebook} páginas")

Notebook 01 convertido: 9 páginas


## 5. Generación del PDF

Primero se arman la portada y las capturas, y al final se les une el notebook 01.

In [4]:
def nueva_pagina(pdf, etapa, titulo):
    fig = plt.figure(figsize=A4)
    if etapa:
        fig.text(0.05, 0.945, etapa.upper(), fontsize=9, color=NARANJA, fontweight="bold")
    fig.text(0.05, 0.9, titulo, fontsize=19, fontweight="bold", color=TINTA)
    fig.text(0.05, 0.03, f"Práctica S17 · IA Explorer TCS / TecNM (CPFIA) · {AUTOR}", fontsize=8, color=GRIS)
    fig.text(0.95, 0.03, str(pdf.get_pagecount() + 1), fontsize=8, color=GRIS, ha="right")
    return fig


def pagina_captura(pdf, etapa, titulo, clave):
    fig = nueva_pagina(pdf, etapa, titulo)
    ax = fig.add_axes([0.05, 0.07, 0.9, 0.79])
    ax.set_xticks([]), ax.set_yticks([])
    ruta = buscar_captura(clave)
    if ruta is None:
        ax.set_facecolor("#F4F6F8")
        ax.text(0.5, 0.5, f"Captura pendiente:\noutputs/capturas/{clave}.png", ha="center", va="center",
                fontsize=12, color="red", transform=ax.transAxes)
    else:
        ax.imshow(mpimg.imread(ruta))
        ax.set_aspect("equal")
        for s in ax.spines.values():
            s.set_edgecolor("#C9D3DD")
    pdf.savefig(fig)
    plt.close(fig)

In [5]:
PARTE_A = OUTPUT_DIR / "_parte_a.pdf"

with PdfPages(PARTE_A) as pdf:
    fig = nueva_pagina(pdf, None, "Calculadora en Python")
    fig.text(0.05, 0.86, f"Práctica 2 · Módulo 2 · Sesión 17, Python para Análisis de Datos · {AUTOR} · {FECHA}",
             fontsize=10.5, color="#5A6773", style="italic")
    fig.text(0.05, 0.74, "Temas de la sesión y dónde los usé en la calculadora",
             fontsize=12, fontweight="bold", color=TINTA)
    ax = fig.add_axes([0.05, 0.26, 0.9, 0.44])
    ax.axis("off")
    temas = [
        ["Variables y tipos de datos", "int, float, str y bool; type() para revisarlos"],
        ["Operadores", "+  -  *  /  //  %  ** en las siete operaciones; comparación y lógicos para validar"],
        ["Listas", "Casos de prueba y el historial de operaciones"],
        ["Diccionarios", "Cada símbolo asociado a su función; cada operación del historial; conteo por operación"],
        ["Condicionales y ciclos", "if para validar, for para las pruebas, while para la calculadora"],
        ["Funciones", "Una función por operación, más convertir_numero y calcular"],
        ["Errores comunes", "TypeError, ValueError y ZeroDivisionError manejados con try / except"],
    ]
    tabla = ax.table(cellText=temas, colLabels=["Tema", "Dónde lo usé"], colWidths=[0.28, 0.72],
                     bbox=[0, 0, 1, 1], cellLoc="left")
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(10)
    for (r, _), celda in tabla.get_celld().items():
        celda.set_edgecolor("#C9D3DD")
        if r == 0:
            celda.set_facecolor(AZUL)
            celda.get_text().set_color("white")
            celda.get_text().set_fontweight("bold")
    fig.text(0.05, 0.12, "Herramientas: Python 3.13 y notebooks de Jupyter en VS Code, sin librerías externas. "
             "El notebook completo está al final del reporte.", fontsize=9, color="#5A6773")
    pdf.savefig(fig)
    plt.close(fig)

    for etapa, titulo, clave in CAPTURAS:
        pagina_captura(pdf, etapa, titulo, clave)

writer = PdfWriter()
for parte in (PARTE_A, NOTEBOOK_PDF):
    writer.append(str(parte))
with open(PDF_OUT, "wb") as f:
    writer.write(f)
writer.close()
for temporal in (PARTE_A, NOTEBOOK_PDF, NOTEBOOK_HTML):
    temporal.unlink()

print(f"PDF guardado: {PDF_OUT.relative_to(BASE_DIR)} ({len(PdfReader(PDF_OUT).pages)} páginas)")
print("Todas las capturas están incluidas." if faltan == 0 else f"Capturas pendientes: {faltan}")

PDF guardado: outputs/S17_Calculadora_Python.pdf (14 páginas)
Todas las capturas están incluidas.
